<a href="https://colab.research.google.com/github/Ymin-2/ESAA/blob/main/ESAA_OB_WEEK04_1_review4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**주제**

항만 內 선박 대기 시간 예측을 위한 선박 항차 데이터 분석 AI 알고리즘 개발  

https://dacon.io/competitions/official/236158/overview/description

##**데이터**

- train.csv
  - Rows : 391,939개
  - SAMPLE_ID : 학습 샘플의 고유 ID
  - [수정] 유가 정보 Feature Drop ['DUBAI', 'BRENT', 'WTI', 'BDI_ADJ'] (이전 버전 데이터 사용 시 실격)
  - Feature 정보는 [https://dacon.io/competitions/official/236158/talkboard/409670?page=1&dtype=recent ]에서 참조
  - CI_HOUR : 대기시간

- test.csv
  - Rows : 220,491개
  - [수정] 유가 정보 Feature Drop ['DUBAI', 'BRENT', 'WTI', 'BDI_ADJ'] (이전 버전 데이터 사용 시 실격)
  - SAMPLE_ID : 추론 샘플의 고유 ID

- sample_submission.csv
  - Rows : 220,491개
  - SAMPLE_ID : 추론 샘플의 고유 ID
  - CI_HOUR : 예측한 대기시간

##**코드리뷰**

[날짜 파생변수 생성]
- ATA -> datetime으로 바꿈
- year, month, day, hour, minute, weekday 만들기

[전처리]
- 도메인 기반 전처리
- 피처 엔지니어링
- 불필요한 컬럼 제거
- 범주형 변수 Label Encoding
- weekday를 평일/주말 이진값으로 단순화
- 타깃 로그변환
- 각각 따로 StandardScaler

[Feature importance 기반 변수 선택]
- train_0, train_1에 대해 XGBoost를 돌린 후 feature importance를 보고 dist==0에서는 threshold 0.004, dist!=0에서는 threshold 0.012 이하 변수 제거

[하이퍼파라미터 탐색 + StratifiedKFold 검증]
- GridSearchCV로 하이퍼파라미터 탐색
- StratifiedKFold로 fold를 나눠 검증
- 각 fold마다 test 예측
- fold 예측 평균으로 최종 제출 예측 생성

[예측 후처리]
- dist==0 예측 중 너무 작은 값은 0으로 보정
- test_0, test_1 각각에 예측 붙이기
- 다시 합치고 SAMPLE_ID 순으로 정렬

##**배울점**

- 도메인 지식을 반영한 피처 엔지니어링
- 데이터를 분리해서 학습 (DIST == 0, DIST != 0)
- SPLIT_CRETERION을 만들어서 stratify에 넣음 -> 항만/선종/거리구간 조합이 fold마다 비슷하도록 설계
- 예측 후처리

In [ ]:
# 피처 엔지니어링
# 깊이
def pre_3_ratio_under_water(data):

    data['ratio_under_water'] = 0

    cond = (data['DRAUGHT'] != 0) & (data['DEPTH'] != 0)

    data.loc[cond, 'ratio_under_water'] = np.round(data.loc[cond, 'DRAUGHT'] / data.loc[cond, 'DEPTH'], 2)
    data.loc[~cond, 'ratio_under_water'] = np.round((data.loc[~cond, 'DRAUGHT']+1) / (data.loc[~cond, 'DEPTH']+1), 2)

    return data


# 흘수
def pre_4_ratio_under_water_DEADWEIGHT(data):

    data['ratio_under_water_DEAD'] = 0

    cond = (data['DRAUGHT'] != 0) & (data['DEPTH'] != 0)

    data.loc[cond, 'ratio_under_water_DEAD'] = np.round(data.loc[cond, 'DRAUGHT'] / data.loc[cond, 'DEPTH'], 2) * data.loc[cond, 'DEADWEIGHT']
    data.loc[~cond, 'ratio_under_water_DEAD'] = np.round((data.loc[~cond, 'DRAUGHT']+1) / (data.loc[~cond, 'DEPTH']+1), 2) * data.loc[~cond, 'DEADWEIGHT']

    return data


# 구간화
def pre_6_dist_binding(data):

    dist_l = list()

    for d in data['DIST']:

        if d < 75:
            dist_l.append(str('00'))
        elif d < 150:
            dist_l.append(str('01'))
        elif d < 175:
            dist_l.append(str('02'))
        else:
            dist_l.append(str('03'))

    data['DIST_binded'] = dist_l

    return data

In [ ]:
# 데이터 분리
# SPLIT : DIST 0 or 1
train_0 = train[train['DIST'] == 0]
train_1 = train[train['DIST'] != 0]
train_1_process_outlier = train[(train['DIST'] != 0) & (train['CI_HOUR'] <= 1000)]

test_0 = test[test['DIST'] == 0]
test_1 = test[test['DIST'] != 0]

print(f'DIST = 0 : {train_0.shape}')
print(f'DIST != 0 : {train_1.shape}')

In [ ]:
# split_creterion
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, test_index) in enumerate(skf.split(train_reduced, SPLIT_CRETERION)):
    train_set = train_reduced.iloc[train_index]
    valid_set = train_reduced.iloc[test_index]

    x_train, y_train = train_set, Y_train_0.iloc[train_index]
    x_val, y_val = valid_set, Y_train_0.iloc[test_index]

In [ ]:
# 예측 후처리
preds = model.predict(test_0_reduced)
preds = np.where(preds < 0, 0, preds)

final_predictions_0[final_predictions_0 < 0.1] = 0